Naïve Bayes Model 2: CategoricalNB
David Beas, Jorge

https://www.kaggle.com/datasets/uciml/mushroom-classification

In [ ]:
# Import all required libraries
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.naive_bayes import CategoricalNB
from sklearn import metrics

In [ ]:
# 1. Load the dataset
df = pd.read_csv('mushrooms.csv')

# 2. Select Features
# To make user predictions reasonable, we select the top 5 most visual/identifiable features.
features = ['cap-shape', 'cap-color', 'odor', 'bruises', 'habitat']
X = df[features].copy()

# The target is 'class' (e = edible, p = poisonous)
y = df['class'].copy()

# Show the first few rows
print(f"Dataset size: {df.shape}")
X.head()

Dataset size: (8124, 23)


,cap-shape,cap-color,odor,bruises,habitat
0,x,n,p,t,u
1,x,y,a,t,g
2,b,w,l,t,m
3,x,w,p,t,u
4,x,g,n,f,g


In [ ]:
# 3. Encode Categorical Variables into numbers
# CategoricalNB requires inputs to be integers, NOT text.
# IMPORTANT: We DO NOT use StandardScaler here. CategoricalNB relies on counting
# distinct categories, not measuring geometric distance! Scaling would break the math.

le_dict = {}

# Encode all 5 features
for col in X.columns:
    le = LabelEncoder()
    X[col] = le.fit_transform(X[col])
    le_dict[col] = le  # Save the encoder so we can use it for user inputs later

# Encode Target Variable (0 = Edible, 1 = Poisonous)
le_target = LabelEncoder()
y = le_target.fit_transform(y)

print("Categorical features successfully encoded to integers!")

Categorical features successfully encoded to integers!


In [ ]:
# 4. Split the data into Training (75%) and Testing (25%) sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)

# 5. Train the Categorical Naïve Bayes model
model_cat = CategoricalNB()
model_cat.fit(X_train, y_train)

# Make predictions on the testing data
y_pred = model_cat.predict(X_test)

In [ ]:
# 6. Run Reliability Metrics (Accuracy, Precision, Recall, F1)
accuracy = metrics.accuracy_score(y_test, y_pred)
precision = metrics.precision_score(y_test, y_pred)
recall = metrics.recall_score(y_test, y_pred)
f1 = metrics.f1_score(y_test, y_pred)

print("--- Naïve Bayes: CategoricalNB ---")
print(f"Accuracy:  {accuracy * 100:.2f}%")
print(f"Precision: {precision:.3f}")
print(f"Recall:    {recall:.3f}")
print(f"F1-Score:  {f1:.3f}")

--- Naïve Bayes: CategoricalNB ---
Accuracy:  98.57%
Precision: 1.000
Recall:    0.971
F1-Score:  0.985


In [ ]:
# 7. User input loop for new predictions
def predict_mushroom():
    while True:
        try:
            print("\n--- Mushroom Edibility Predictor ---")
            print("Please enter the single-letter code for each feature:")

            shape = input("Cap Shape (b=bell, c=conical, x=convex, f=flat, k=knobbed, s=sunken): ").lower()
            color = input("Cap Color (n=brown, b=buff, c=cinnamon, g=gray, r=green, p=pink, u=purple, e=red, w=white, y=yellow): ").lower()
            odor = input("Odor (a=almond, l=anise, c=creosote, y=fishy, f=foul, m=musty, n=none, p=pungent, s=spicy): ").lower()
            bruises = input("Bruises easily? (t=yes, f=no): ").lower()
            habitat = input("Habitat (g=grasses, l=leaves, m=meadows, p=paths, u=urban, w=waste, d=woods): ").lower()

            # Encode the user inputs using the exact same LabelEncoders from training
            user_data = pd.DataFrame([[shape, color, odor, bruises, habitat]], columns=features)

            for col in features:
                user_data[col] = le_dict[col].transform(user_data[col])

            # Make the prediction
            prediction = model_cat.predict(user_data)[0]
            probabilities = model_cat.predict_proba(user_data)[0]

            print("\n-----------------------------------------")
            if prediction == 1:
                print(f"☠️ PREDICTION: POISONOUS (Confidence: {probabilities[1]*100:.1f}%)")
                print("Do NOT eat this mushroom!")
            else:
                print(f"🍽️ PREDICTION: EDIBLE (Confidence: {probabilities[0]*100:.1f}%)")
                print("This mushroom appears safe to eat.")
            print("-----------------------------------------")

            run_again = input("Would you like to test another mushroom? (yes/no): ")
            if run_again.lower() != 'yes':
                break

        except ValueError:
            print("\nError: You entered a letter code that the model doesn't recognize. Please try again.")

# Run the prediction function
predict_mushroom()


--- Mushroom Edibility Predictor ---
Please enter the single-letter code for each feature:
Cap Shape (b=bell, c=conical, x=convex, f=flat, k=knobbed, s=sunken): b
Cap Color (n=brown, b=buff, c=cinnamon, g=gray, r=green, p=pink, u=purple, e=red, w=white, y=yellow): g
Odor (a=almond, l=anise, c=creosote, y=fishy, f=foul, m=musty, n=none, p=pungent, s=spicy): n
Bruises easily? (t=yes, f=no): f
Habitat (g=grasses, l=leaves, m=meadows, p=paths, u=urban, w=waste, d=woods): l

-----------------------------------------
🍽️ PREDICTION: EDIBLE (Confidence: 97.0%)
This mushroom appears safe to eat.
-----------------------------------------
Would you like to test another mushroom? (yes/no): no


For our second model, we sought to predict whether a wild mushroom is edible or poisonous based purely on its visual and physical characteristics (such as its cap shape, color, odor, and habitat). Because this dataset is made entirely of distinct, labeled categories rather than continuous numerical measurements, we utilized the Categorical Naïve Bayes algorithm. This specific version of Naïve Bayes is mathematically optimized to calculate the probability of an outcome by looking at combinations of distinct traits. Importantly, because CategoricalNB relies on counting occurrences of categories rather than measuring geometric distance between points, we purposefully bypassed the "Standard Scaler" step used in previous algorithms, as scaling categorical data would break the underlying math.

Even after streamlining the model to look at only the 5 most easily identifiable features, it achieved an outstanding accuracy of 98.5%, alongside near-perfect Precision and Recall scores. The model heavily prioritized features like "Odor"—quickly learning that foul, pungent, or fishy smells almost exclusively point to a poisonous classification. From a practical standpoint, this highly accurate and lightweight model could easily be deployed as a backend algorithm for a mobile hiking or foraging app, allowing users to input a few quick visual characteristics to determine if a wild fungi is safe to consume.